# LLM Fine-tuning - Turkish Legal QA (A100)

**Proje:** CENG493 Turkish Legal RAG  
**Adim:** 4/5 - LLM Fine-tuning  
**GPU:** A100 (40GB)  
**Onceki:** Reranker (R@5=82.0%, F1=0.305, EM=0.033)

**Proje gereksinimleri:**
- Instruction tuning
- LoRA or QLoRA
- Supervised fine-tuning
- Retrieval-aware prompting

**Model:** Qwen2.5-7B-Instruct + QLoRA  
**Egitim verisi:** Kaggle `turkish_law_dataset.csv` — ayni satirdaki `context` + `soru` + `cevap` (corpus ile hizali; FAISS tahmini yok)  
**Beklenen etki:** Daha tutarli baglam-cevap; gold test ile hizalamayi ayrica iyilestirebilirsiniz

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl datasets sentence-transformers faiss-cpu rank_bm25 snowballstemmer rouge-score nltk

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, re, math, random
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/493_project")

import sys
sys.path.insert(0, str(PROJECT_DIR))
from evaluation.metrics_utils import (
    RAG_SYSTEM_PROMPT,
    format_context_blocks_for_llm,
    format_sft_assistant_with_sources,
    aggregate_qa_row,
    retrieval_hit,
)

chunks = []
with open(PROJECT_DIR / "data/processed/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))
chunk_texts = [c["text"] for c in chunks]

with open(PROJECT_DIR / "evaluation/gold_test_set.json", "r", encoding="utf-8") as f:
    gold_questions = json.load(f)
answerable_qs = [q for q in gold_questions if q.get("answerable_from_corpus", True)]

with open(PROJECT_DIR / "evaluation/baseline_results.json", "r", encoding="utf-8") as f:
    baseline_results = json.load(f)

_rr_path = PROJECT_DIR / "evaluation" / "reranker_results.json"
if _rr_path.is_file():
    with open(_rr_path, "r", encoding="utf-8") as f:
        reranker_results = json.load(f)
    print(f"Reranker F1 (03 ciktisi): {reranker_results['metrics_tuned_reranker']['f1']:.4f}")
else:
    reranker_results = None
    print(
        "[UYARI] reranker_results.json yok — tam sifirdan basliyorsan normal. "
        "Karsilastirma tablosunda '+ Reranker' sutunu NaN olur; 03 bittikten sonra bu dosya olusur."
    )

KG_CSV = PROJECT_DIR / "data/raw/turkish_law_dataset.csv"
kg_raw = pd.read_csv(KG_CSV)

print(f"Chunks: {len(chunks)}")
print(f"Answerable Qs: {len(answerable_qs)}")
print(f"Kaggle raw rows: {len(kg_raw)}")
print(f"Baseline F1: {baseline_results['metrics']['f1']:.4f}")

## 1. Retrieval-aware egitim verisi (Kaggle, hizali baglam)

`prepare_corpus.py` ile ayni kaynak: **Kaggle CSV**deki her satirda zaten `soru`, `cevap` ve o ciftle eslesen **`context`** var.  
Satir `context_clean`, `data/processed/chunks.jsonl` ile **tam metin eslestirilirse** user mesaji `format_context_blocks_for_llm` ( `[chunk_id=...]` ) ile yazilir; assistant hedefi **cevap + Kaynaklar + chunk_id** olur (`format_sft_assistant_with_sources`). Eslesmezse (nadir) eski duz baglam + duz cevap.

- `Score >= 8`, bos alanlar elenir, `context` uzunluk esigi (`prepare_corpus` ile uyumlu)
- **Gold test** sorulari (tam metin eslesmesi) egitimden cikarilir
- Ayni `context` tekrarlari tek ornekte birlestirilir
- `max_length=1024` nedeniyle user baglami **MAX_CONTEXT_CHARS** ile kirpilir (chunk eslesmesinde de ayni kirpma)

```
System: RAG_SYSTEM_PROMPT
User: Baglam: [chunk_id=...] Kaynak: ...  Soru: ...
Assistant: [cevap]\n\nKaynaklar:\n- [chunk_id=...] ...
```

In [ ]:
# prepare_corpus.py ile ayni kolonlar ve Score filtresi
REQUIRED = ["soru", "cevap", "veri türü", "kaynak", "context", "Score"]
missing = [c for c in REQUIRED if c not in kg_raw.columns]
if missing:
    raise ValueError(f"Kaggle CSV eksik kolon: {missing} | Mevcut: {list(kg_raw.columns)}")

MIN_SCORE = 8
MIN_CONTEXT_LEN = 100
MAX_CONTEXT_CHARS = 3200  # max_length=1024 ile uyum; gerekirse artirin ve MAX_SEQ_LEN'i yukseltin


def normalize_whitespace(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\ufeff", " ").replace("\xa0", " ")
    lines = [line.strip() for line in text.split("\n")]
    cleaned_lines = []
    previous_blank = False
    for line in lines:
        is_blank = line == ""
        if is_blank and previous_blank:
            continue
        cleaned_lines.append(line)
        previous_blank = is_blank
    text = "\n".join(cleaned_lines).strip()
    text = re.sub(r"[ \t]+", " ", text)
    return text


kg = kg_raw.copy()
kg["Score"] = pd.to_numeric(kg["Score"], errors="coerce")
kg = kg[kg["Score"] >= MIN_SCORE].copy()
kg["soru"] = kg["soru"].astype(str).apply(normalize_whitespace)
kg["cevap"] = kg["cevap"].astype(str).apply(normalize_whitespace)
kg["context_clean"] = kg["context"].astype(str).apply(normalize_whitespace)
kg["kaynak_clean"] = kg["kaynak"].astype(str).apply(normalize_whitespace)
kg = kg[
    (kg["soru"].str.len() > 0)
    & (kg["cevap"].str.len() > 0)
    & (kg["context_clean"].str.len() >= MIN_CONTEXT_LEN)
].copy()

gold_q_set = {q["soru"].lower().strip() for q in gold_questions}
before_gold = len(kg)
kg = kg[~kg["soru"].str.lower().str.strip().isin(gold_q_set)].copy()
print(f"Gold overlap cikarildi: {before_gold} -> {len(kg)}")

# Ayni context altinda farkli sorular gecerli; sadece tamamen ayni satirlari at
kg = kg.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Kaggle egitim satirlari (tam satir dedup + filtre sonrasi): {len(kg)}")

SYSTEM_PROMPT = RAG_SYSTEM_PROMPT

chunk_by_context = {}
for _c in chunks:
    _k = normalize_whitespace(_c.get("text", ""))
    if _k:
        chunk_by_context[_k] = _c

def _clip_ctx(t: str) -> tuple[str, bool]:
    if len(t) <= MAX_CONTEXT_CHARS:
        return t, False
    return (
        t[:MAX_CONTEXT_CHARS].rstrip() + "\n\n[Baglam uzunlugu nedeniyle kisaltilmistir.]",
        True,
    )


train_data = []
truncated = 0
n_sft_rich = 0
n_sft_plain = 0
for _, row in kg.iterrows():
    ctx_full = row["context_clean"]
    kaynak = row["kaynak_clean"]
    rec = chunk_by_context.get(ctx_full)

    if rec:
        txt = rec.get("text", ctx_full)
        ctx_vis, was_t = _clip_ctx(txt)
        if was_t:
            truncated += 1
        rec_msg = {**rec, "text": ctx_vis}
        user_msg = f"Baglam:\n{format_context_blocks_for_llm([rec_msg], max_chunks=1)}\n\nSoru: {row['soru']}"
        assistant = format_sft_assistant_with_sources(
            row["cevap"],
            str(rec["chunk_id"]),
            str(rec.get("source", kaynak)),
            ctx_vis,
        )
        n_sft_rich += 1
    else:
        ctx_user, was_t = _clip_ctx(ctx_full)
        if was_t:
            truncated += 1
        context_block = f"[{kaynak}]\n{ctx_user}" if kaynak else ctx_user
        user_msg = f"Baglam:\n{context_block}\n\nSoru: {row['soru']}"
        assistant = row["cevap"]
        n_sft_plain += 1

    train_data.append({
        "system": SYSTEM_PROMPT,
        "user": user_msg,
        "assistant": assistant,
    })

if truncated:
    print(f"Uyari: {truncated} ornekte context {MAX_CONTEXT_CHARS} karaktere kirpildi.")

random.seed(42)
random.shuffle(train_data)

print(f"Training examples: {len(train_data)}")
print(f"  chunk eslesmeli (Kaynaklar+chunk_id): {n_sft_rich}, dusen (duz cevap): {n_sft_plain}")
print("\n--- Ornek ---")
print(f"User (ilk 280 karakter): {train_data[0]['user'][:280]}...")
print(f"Assistant (ilk 320 karakter): {train_data[0]['assistant'][:320]}")

In [ ]:
# train_data yukaridaki hucrede uretildi; bu hucre atlanabilir.
pass

In [ ]:
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

save_path = PROJECT_DIR / "data/processed/llm_train_data.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)
print(f"Saved {len(train_data)} examples -> {save_path}")

## 2. QLoRA Fine-tuning

**Qwen2.5-7B-Instruct** uzerine QLoRA:
- 4-bit NF4 quantization
- LoRA rank=16, alpha=32
- batch=8, grad_accum=4 (efektif 32)
- lr=**5e-5** (patlamaya karsi dusuk), warmup=250, **max_grad_norm=0.3**
- **num_train_epochs=1** — 1 epoch sorunsuz biterse ayni ayarlarla 2 epoch tekrarlanabilir
- **Her 10 adimda** train loss loglanir; **her 50 adimda** checkpoint
- **Early stop:** `global_step >= warmup_steps` sonrasi train loss > 2.3 (baslangic yuksek loss yoksayilir)
- max_length=1024

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
def format_chat(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(train_data)
dataset = dataset.map(format_chat)

token_lengths = [len(tokenizer(t["text"]).input_ids) for t in dataset.select(range(200))]
print(f"Sample token lengths: min={min(token_lengths)}, median={sorted(token_lengths)[100]}, max={max(token_lengths)}")
print(f"Dataset: {len(dataset)} examples")
print(f"\nExample text (first 300 chars):")
print(dataset[0]["text"][:300])

In [ ]:
from trl import SFTConfig
from transformers import TrainerCallback


class StopOnTrainLossSpike(TrainerCallback):
    """Gradient patlamasi: loss dusup sonra tekrar esigi astiginda durdur.
    Ilk adimlarda loss 4-6 normaldir; min_global_step oncesi kontrol YOK."""

    def __init__(
        self,
        threshold: float = 2.3,
        consecutive_high_logs: int = 1,
        min_global_step: int = 250,
    ):
        self.threshold = threshold
        self.consecutive_high_logs = consecutive_high_logs
        self.min_global_step = min_global_step
        self._streak = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return control
        if state.global_step < self.min_global_step:
            return control
        loss = float(logs["loss"])
        if loss > self.threshold:
            self._streak += 1
            print(
                f"  [spike-watch] step={state.global_step} loss={loss:.4f} > {self.threshold} "
                f"(ust uste yuksek log: {self._streak}/{self.consecutive_high_logs})"
            )
            if self._streak >= self.consecutive_high_logs:
                print(
                    f"\n[EARLY STOP] Train loss patlamasi. Son iyi checkpoint: "
                    f"{args.output_dir}/checkpoint-* veya resume_from_checkpoint ile devam."
                )
                control.should_training_stop = True
        else:
            self._streak = 0
        return control


MAX_SEQ_LEN = 1024

training_args = SFTConfig(
    output_dir=str(PROJECT_DIR / "models" / "qwen-legal-qlora"),
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=250,
    max_grad_norm=0.3,
    bf16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=5,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    max_length=MAX_SEQ_LEN,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    callbacks=[
        StopOnTrainLossSpike(
            threshold=2.3,
            consecutive_high_logs=1,
            min_global_step=training_args.warmup_steps,
        )
    ],
)

print(f"Training config (1 epoch, LR=5e-5, max_grad_norm=0.3, ara checkpoint):")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  LR: {training_args.learning_rate} | warmup_steps: {training_args.warmup_steps} | max_grad_norm: {training_args.max_grad_norm}")
print(f"  Max seq len: {MAX_SEQ_LEN} | log every {training_args.logging_steps} steps | save every {training_args.save_steps} steps")
print(f"  Early stop: step>={training_args.warmup_steps} iken loss > 2.3")
print(f"  Steps/epoch: {len(trainer.get_train_dataloader())}")
print(f"  Total steps: {len(trainer.get_train_dataloader()) * int(training_args.num_train_epochs)}")

In [ ]:
print("Starting QLoRA fine-tuning...")
trainer.train()

lora_save_path = str(PROJECT_DIR / "models" / "qwen-legal-qlora" / "final")
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)

print(f"\nFine-tuning complete!")
print(f"LoRA adapters saved -> {lora_save_path}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Evaluation

Fine-tuned modeli retrieval pipeline ile birlestirip degerlendiriyoruz.  
Pipeline: Hybrid Retrieve (top-20) -> Rerank -> Top-5 -> Fine-tuned LLM -> Answer

In [ ]:
del trainer
torch.cuda.empty_cache()

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
ft_model = PeftModel.from_pretrained(base_model, lora_save_path)
ft_model.eval()

ft_tokenizer = AutoTokenizer.from_pretrained(lora_save_path)
print(f"Fine-tuned model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import snowballstemmer
from rank_bm25 import BM25Okapi

embed_model = SentenceTransformer(str(PROJECT_DIR / "models" / "bge-m3-legal-v2"), device="cuda")

tuned_vecs = np.load(str(PROJECT_DIR / "models" / "embedding_tuned_v2" / "dense_vecs.npy"))
tuned_index = faiss.IndexFlatIP(tuned_vecs.shape[1])
tuned_index.add(tuned_vecs)

RERANKER_DIR = PROJECT_DIR / "models" / "reranker-legal"
BASE_RERANKER = "BAAI/bge-reranker-v2-m3"
if RERANKER_DIR.is_dir() and (RERANKER_DIR / "config.json").is_file():
    print(f"Reranker (fine-tuned): {RERANKER_DIR}")
    reranker = CrossEncoder(
        str(RERANKER_DIR.resolve()),
        max_length=512,
        device="cuda",
        automodel_args={"local_files_only": True},
        tokenizer_args={"local_files_only": True},
    )
else:
    print(
        f"[UYARI] Fine-tuned reranker yok ({RERANKER_DIR}). "
        f"Hub'dan base model yukleniyor: {BASE_RERANKER}\n"
        "Tam pipeline icin 03_reranker.ipynb calistirip Drive'a kaydedin."
    )
    reranker = CrossEncoder(BASE_RERANKER, max_length=512, device="cuda")

tr_stemmer = snowballstemmer.stemmer('turkish')
TURKISH_STOPWORDS = {
    "bir", "bu", "de", "da", "ve", "ile", "için", "olan", "olarak",
    "veya", "ya", "hem", "ne", "her", "o", "şu", "ki", "mi", "mu",
    "mı", "mü", "ise", "den", "dan", "ten", "tan", "dir", "dır",
    "dür", "dur", "tir", "tır", "tur", "tür", "gibi", "kadar",
    "sonra", "önce", "üzere", "göre", "ait", "dair", "en", "çok",
    "var", "yok", "olan", "olup", "ancak", "fakat", "ama",
}

def tokenize_tr(text: str) -> list:
    tokens = re.findall(r'\w+', text.lower())
    tokens = [t for t in tokens if t not in TURKISH_STOPWORDS and len(t) > 1]
    return tr_stemmer.stemWords(tokens)

tokenized_chunks = [tokenize_tr(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

print(f"Retrieval pipeline loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def _make_result(idx, rank, score):
    c = chunks[idx]
    return {
        "rank": rank, "score": score,
        "chunk_id": c["chunk_id"], "source": c["source"],
        "article_title": c.get("article_title", ""),
        "article_key": c.get("article_key", ""),
        "text": c["text"],
    }


def retrieve_hybrid(query, top_k=20, alpha=0.8):
    q_vec = embed_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")
    dense_scores = np.dot(tuned_vecs, q_vec.T).flatten()
    bm25_scores = bm25.get_scores(tokenize_tr(query))
    d_min, d_max = dense_scores.min(), dense_scores.max()
    dense_norm = (dense_scores - d_min) / (d_max - d_min + 1e-9)
    bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)
    combined = alpha * dense_norm + (1 - alpha) * bm25_norm
    top_indices = np.argsort(combined)[::-1][:top_k]
    return [_make_result(int(idx), r+1, float(combined[idx])) for r, idx in enumerate(top_indices)]


def rerank_results(query, retrieved, top_k=5):
    pairs = [(query, r["text"]) for r in retrieved]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(scores, retrieved), key=lambda x: -x[0])
    return [{**r, "rank": i+1, "rerank_score": float(s)} for i, (s, r) in enumerate(ranked[:top_k])]


def rag_answer_ft(question, retrieved):
    context = format_context_blocks_for_llm(retrieved, max_chunks=5)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Baglam:\n{context}\n\nSoru: {question}"},
    ]
    text = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ft_tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=384, do_sample=False)
    return ft_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


print("\nTest:")
test_ret = retrieve_hybrid("Egemenlik kime aittir?", top_k=20)
test_rr = rerank_results("Egemenlik kime aittir?", test_ret, top_k=5)
test_ans = rag_answer_ft("Egemenlik kime aittir?", test_rr)
print(f"Soru: Egemenlik kime aittir?")
print(f"Cevap: {test_ans}")

In [ ]:
import nltk

for _p in ("punkt", "punkt_tab"):
    try:
        nltk.download(_p, quiet=True)
    except Exception:
        pass

print("Eval: aggregate_qa_row + retrieval_hit -> evaluation.metrics_utils")

In [ ]:
from tqdm import tqdm

results_ft = []

for q in tqdm(answerable_qs, desc="Evaluating"):
    candidates = retrieve_hybrid(q["soru"], top_k=20)
    reranked = rerank_results(q["soru"], candidates, top_k=5)
    rh = retrieval_hit(reranked, q)

    answer = rag_answer_ft(q["soru"], reranked)
    qa = aggregate_qa_row(answer, q["cevap"], reranked, q)
    pred_body = qa.pop("pred_body")

    results_ft.append({
        "id": q["id"], "soru": q["soru"],
        "gold_cevap": q["cevap"], "kaynak": q["kaynak"],
        "difficulty": q["difficulty"], "type": q["type"],
        **rh,
        "pred_cevap": answer,
        "pred_body": pred_body,
        **qa,
    })

print(f"\nDone: {len(results_ft)} questions evaluated")

In [ ]:
df_ft = pd.DataFrame(results_ft)
bl = baseline_results["metrics"]
if reranker_results is not None:
    rr = reranker_results["metrics_tuned_reranker"]
else:
    rr = {f"recall@{k}": float("nan") for k in [1, 3, 5, 10]}
    rr.update({
        "mrr": float("nan"), "ndcg@10": float("nan"),
        "f1": float("nan"), "em": float("nan"), "bleu": float("nan"),
        "rouge1": float("nan"), "rouge2": float("nan"), "rougeL": float("nan"),
        "faithfulness_token_recall": float("nan"),
        "citation_precision_retrieved": float("nan"),
        "citation_recall_gold_mean": float("nan"),
    })

print("=" * 75)
print("  LLM FINE-TUNING SONUCLARI")
print("=" * 75)

print(f"\n{'Metric':<12} {'Baseline':>10} {'+ Reranker':>12} {'+ LLM FT':>12} {'Delta(BL)':>10}")
print("-" * 58)
for k in [1, 3, 5, 10]:
    b = bl[f"recall@{k}"]
    r = rr[f"recall@{k}"]
    ft = df_ft[f"hit@{k}"].mean()
    print(f"Recall@{k:<2d}    {b:>9.4f}  {r:>11.4f}  {ft:>11.4f}  {ft-b:>+9.4f}")

ft_mrr = df_ft["mrr"].mean()
ft_ndcg = df_ft["ndcg@10"].mean()
bl_ndcg = bl.get("ndcg@10", float("nan"))
delta_ndcg = (
    (ft_ndcg - bl_ndcg)
    if not (isinstance(bl_ndcg, float) and bl_ndcg != bl_ndcg)
    else float("nan")
)
print(f"{'MRR':<12} {bl['mrr']:>9.4f}  {rr['mrr']:>11.4f}  {ft_mrr:>11.4f}  {ft_mrr-bl['mrr']:>+9.4f}")
print(f"{'nDCG@10':<12} {bl_ndcg:>9.4f}  {rr['ndcg@10']:>11.4f}  {ft_ndcg:>11.4f}  {delta_ndcg:>+9.4f}")

ft_f1 = df_ft["f1"].mean()
ft_em = df_ft["em"].mean()
ft_r1 = df_ft["rouge1"].mean()
ft_r2 = df_ft["rouge2"].mean()
ft_rL = df_ft["rougeL"].mean()
ft_bleu = df_ft["bleu"].mean()
ft_ff = df_ft["faithfulness_token_recall"].mean()
ft_cp = df_ft["citation_precision_retrieved"].mean()
_ft_cr = df_ft.loc[df_ft["citation_recall_gold"] >= 0, "citation_recall_gold"]
ft_cr = float(_ft_cr.mean()) if len(_ft_cr) else float("nan")

def _gv(d, k):
    v = d.get(k, float("nan"))
    return v if isinstance(v, (int, float)) else float("nan")

print(f"\n{'Metric':<12} {'Baseline':>10} {'+ Reranker':>12} {'+ LLM FT':>12} {'Delta(BL)':>10}")
print("-" * 58)
print(f"{'F1':<12} {bl['f1']:>9.4f}  {rr['f1']:>11.4f}  {ft_f1:>11.4f}  {ft_f1-bl['f1']:>+9.4f}")
print(f"{'EM':<12} {bl['em']:>9.4f}  {rr['em']:>11.4f}  {ft_em:>11.4f}  {ft_em-bl['em']:>+9.4f}")
print(f"{'BLEU':<12} {_gv(bl,'bleu'):>9.4f}  {_gv(rr,'bleu'):>11.4f}  {ft_bleu:>11.4f}  {ft_bleu-_gv(bl,'bleu'):>+9.4f}")
print(f"{'ROUGE-1':<12} {_gv(bl,'rouge1'):>9.4f}  {_gv(rr,'rouge1'):>11.4f}  {ft_r1:>11.4f}  {ft_r1-_gv(bl,'rouge1'):>+9.4f}")
print(f"{'ROUGE-2':<12} {_gv(bl,'rouge2'):>9.4f}  {_gv(rr,'rouge2'):>11.4f}  {ft_r2:>11.4f}  {ft_r2-_gv(bl,'rouge2'):>+9.4f}")
print(f"{'ROUGE-L':<12} {_gv(bl,'rougeL'):>9.4f}  {_gv(rr,'rougeL'):>11.4f}  {ft_rL:>11.4f}  {ft_rL-_gv(bl,'rougeL'):>+9.4f}")
print(f"{'Faithful.':<12} {_gv(bl,'faithfulness_token_recall'):>9.4f}  {_gv(rr,'faithfulness_token_recall'):>11.4f}  {ft_ff:>11.4f}  {ft_ff-_gv(bl,'faithfulness_token_recall'):>+9.4f}")
print(f"{'CitePrec':<12} {_gv(bl,'citation_precision_retrieved'):>9.4f}  {_gv(rr,'citation_precision_retrieved'):>11.4f}  {ft_cp:>11.4f}  {ft_cp-_gv(bl,'citation_precision_retrieved'):>+9.4f}")
def _num(x):
    if x is None:
        return float("nan")
    if isinstance(x, (int, float)) and x == x:
        return float(x)
    return float("nan")

print(f"{'CiteRecGold':<12} {_num(bl.get('citation_recall_gold_mean')):>9.4f}  {_num(rr.get('citation_recall_gold_mean')):>11.4f}  {ft_cr:>11.4f}  (n={len(_ft_cr)})")

print(f"\n--- Difficulty ---")
for d in ["easy", "medium", "hard", "very hard"]:
    s = df_ft[df_ft["difficulty"] == d]
    if len(s):
        print(f"  {d:10s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  BLEU={s['bleu'].mean():.3f}  ROUGE-L={s['rougeL'].mean():.3f}  (n={len(s)})")

print(f"\n--- Question Type ---")
for t in sorted(df_ft["type"].unique()):
    s = df_ft[df_ft["type"] == t]
    print(f"  {t:15s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  BLEU={s['bleu'].mean():.3f}  ROUGE-L={s['rougeL'].mean():.3f}  (n={len(s)})")

In [ ]:
import math

def _reranker_metrics_for_json():
    if reranker_results is None:
        return None
    out = {}
    for k, v in rr.items():
        if isinstance(v, float) and math.isnan(v):
            out[k] = None
        else:
            out[k] = float(v) if isinstance(v, (int, float)) else v
    return out

output = {
    "experiment": "llm_finetuning",
    "model": "Qwen/Qwen2.5-7B-Instruct + QLoRA",
    "embedding": "BAAI/bge-m3 (legal fine-tuned)",
    "retrieval": "hybrid_dense_bm25 (alpha=0.8)",
    "reranker": "BAAI/bge-reranker-v2-m3 (legal fine-tuned)",
    "pipeline": "hybrid_top20 -> rerank_top5 -> fine-tuned_LLM",
    "qlora_config": {
        "r": 16, "alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "batch_size": 8, "grad_accum": 4,
        "lr": 5e-5, "epochs": 1,
        "max_seq_length": 1024,
    },
    "train_data_size": len(train_data),
    "num_questions": len(results_ft),
    "metrics": {
        f"recall@{k}": float(df_ft[f"hit@{k}"].mean()) for k in [1,3,5,10]
    } | {
        "mrr": float(ft_mrr), "ndcg@10": float(ft_ndcg),
        "f1": float(ft_f1), "em": float(ft_em),
        "rouge1": float(ft_r1), "rouge2": float(ft_r2), "rougeL": float(ft_rL),
        "bleu": float(ft_bleu),
        "faithfulness_token_recall": float(df_ft["faithfulness_token_recall"].mean()),
        "citation_precision_retrieved": float(df_ft["citation_precision_retrieved"].mean()),
        "citation_recall_gold_mean": float(_ft_cr.mean()) if len(_ft_cr) else None,
    },
    "baseline_metrics": bl,
    "reranker_metrics": _reranker_metrics_for_json(),
    "per_question": results_ft,
}

save_path = PROJECT_DIR / "evaluation" / "llm_finetuning_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {save_path}")
print("\nSonraki adim: Fully Optimized System (05)")